# Data generation

## Step 1 — Import Required Libraries

In this step, we import the necessary Python libraries for the project.

- `pandas`: used to create and manipulate datasets (DataFrames)
- `numpy`: used to generate random data
- `os`: used to manage directories and file paths

These libraries will be used throughout the dataset generation process.

In [1]:
# Data manipulation
import pandas as pd
import numpy as np

# File system
import os

# Optional (pour vérifier / debug)
import matplotlib.pyplot as plt

## Step 2 — Create Data Directories

In this step, we create the folder structure to store the generated datasets.

- `data/v1/`: will contain the first version of the dataset (original schema)
- `data/v2/`: will contain the second version of the dataset (with schema evolution)

The `exist_ok=True` parameter ensures that no error is raised if the folders already exist.

In [2]:
os.makedirs("data/v1", exist_ok=True)
os.makedirs("data/v2", exist_ok=True)

## Step 3 — Define Variables and Distributions

In this step, we define the main parameters and possible values for our dataset.

- `n`: number of rows to generate (dataset size)
- `regions`: possible geographic regions
- `event_types`: types of user actions (e.g., click, view, purchase)
- `device_types`: types of devices (used only in v2)

These lists will be used to randomly generate realistic data distributions.

In [7]:
n = 10000  # number of rows for the first test dataset

regions = ["EU", "US", "ASIA"]
event_types = ["click", "view", "purchase"]
device_types = ["mobile", "desktop"]

## Step 4 — Generate Dataset Version 1 (v1)

In this step, we generate the first version of the dataset (`v1`) using the required schema.

Each column is generated with a specific distribution:

- `ts`: generated using a time range with a fixed frequency (one event per minute)
- `user_id`: randomly generated integers (uniform distribution)
- `region`: randomly selected from predefined categories
- `event_type`: randomly selected user actions
- `value`: random floating-point values between 0 and 1

This dataset represents the baseline version before schema evolution.

In [13]:
df_v1 = pd.DataFrame({
    "ts": pd.date_range(start="2026-01-01", periods=n, freq="min"),
    "user_id": np.random.randint(1, 1000, n),
    "region": np.random.choice(regions, n),
    "event_type": np.random.choice(event_types, n),
    "value": np.random.rand(n)
})

df_v1.head()

,ts,user_id,region,event_type,value
0,2026-01-01 00:00:00,822,US,view,0.603574
1,2026-01-01 00:01:00,144,EU,click,0.948062
2,2026-01-01 00:02:00,861,EU,view,0.273682
3,2026-01-01 00:03:00,378,ASIA,view,0.671145
4,2026-01-01 00:04:00,471,US,purchase,0.398226


## Step 5 — Generate Dataset Version 2 (v2)

In this step, we create the second version of the dataset (`v2`) to simulate schema evolution.

We start by copying the original dataset (`v1`) to keep all existing columns unchanged.

Then, we add a new column:

- `device_type`: randomly assigned values (e.g., "mobile", "desktop")

This follows the recommended schema evolution strategy: adding a new column, which is backward-compatible.

The resulting dataset (`v2`) contains all columns from `v1` plus the new field.

In [9]:
df_v2 = df_v1.copy()
df_v2["device_type"] = np.random.choice(device_types, n)

df_v2.head()

,ts,user_id,region,event_type,value,device_type
0,2026-01-01 00:00:00,669,ASIA,view,0.497965,mobile
1,2026-01-01 00:01:00,129,US,view,0.782730,mobile
2,2026-01-01 00:02:00,903,ASIA,purchase,0.371717,desktop
3,2026-01-01 00:03:00,791,US,click,0.360322,desktop
4,2026-01-01 00:04:00,73,EU,click,0.592079,desktop


## Step 6 — Save Datasets in Parquet Format

In this step, we store both dataset versions (`v1` and `v2`) in Parquet format.

- `df_v1` is saved in `data/v1/data.parquet`
- `df_v2` is saved in `data/v2/data.parquet`

Parquet is a columnar storage format optimized for analytical workloads.  
It provides better performance and compression compared to row-based formats like CSV.

Saving the datasets in Parquet format allows efficient querying and aligns with the project requirements.

In [ ]:
df_v1 = pd.DataFrame({
    # timestamps: random within 30 days
    "ts": pd.to_datetime("2026-01-01") + pd.to_timedelta(
        np.random.randint(0, 60*24*30, n), unit="m"
    ),

    # user_id: skewed activity (few active users, many inactive)
    "user_id": np.random.choice(
        np.arange(1, 1000),
        size=n,
        p=np.random.dirichlet(np.ones(999))
    ),

    # region: more users in EU (example realistic distribution)
    "region": np.random.choice(
        ["EU", "US", "ASIA"],
        size=n,
        p=[0.5, 0.3, 0.2]
    ),

    # event_type: most events are views, fewer clicks, rare purchases
    "event_type": np.random.choice(
        ["view", "click", "purchase"],
        size=n,
        p=[0.7, 0.25, 0.05]
    ),

    # value: skewed positive values (e.g., spending, interaction score)
    "value": np.random.exponential(scale=30, size=n)
})

df_v1.head()

In [ ]:
df_v1.to_parquet("data/v1/data.parquet")
df_v2.to_parquet("data/v2/data.parquet")

print("Datasets saved successfully in Parquet format")

Datasets saved successfully in Parquet format 


## Step 4 — Generate Dataset Version 1 (v1) with Realistic and Justified Distributions

In this step, we generate a synthetic dataset that mimics real-world user behavior patterns.

Each variable is generated based on realistic assumptions:

- `ts` (timestamp):
  Events are randomly distributed over a 30-day period.
  This reflects real-world systems where user activity occurs continuously over time, rather than at fixed intervals.

- `user_id`:
  A skewed distribution is used to simulate user activity imbalance.
  In real platforms (e.g., social media, e-commerce), a small number of users generate most of the activity,
  while many users are less active (long-tail distribution).

- `region`:
  The distribution is non-uniform (EU: 50%, US: 30%, ASIA: 20%).
  This reflects real-world user bases where certain regions dominate depending on the platform.

- `event_type`:
  Events follow realistic proportions:
  - "view" is the most frequent action
  - "click" is less frequent
  - "purchase" is rare
  This reflects typical user behavior funnels in online systems.

- `value`:
  Values follow an exponential distribution.
  This models real-world metrics such as spending or engagement,
  where most values are small and a few are very large (heavy-tailed behavior).

These design choices aim to simulate realistic data distributions for meaningful analysis.